<a href="https://colab.research.google.com/github/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/blob/rag/rag-inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [30]:
PARQUET_PATH = '/content/drive/MyDrive/Progetto-NLP/Branch-rag/collection_ita.parquet'
EMBEDDINGS_PATH = "/content/drive/MyDrive/Progetto-NLP/Branch-rag/embedding_collection_ita"
VECTOR_DB_PATH = "/content/drive/MyDrive/Progetto-NLP/Branch-rag/embedding_collection_ita/db/db_local"
VECTOR_DB_PATH_LOCAL = "/content/db"
DS_PATH="/content/drive/MyDrive/Progetto-NLP/Branch-rag/"
CACHE_DIR = "/content/drive/MyDrive/Progetto-NLP/hf_cache/"

'''
Progetto-NLP/
├─ Branch-rag/
│  ├─ embeddings_collection_ita/
│  │  ├─ status_registry.json   #mancanti vs completati
│  │  ├─ embeddings_chunk_i_i+10k.pkl
│  │  ├─ embeddings_...
│  │  ├─ db/
│  │  │  ├─ db_registry.json   #elementi aggiunti al db

'''

'\nProgetto-NLP/\n├─ Branch-rag/\n│  ├─ embeddings_collection_ita/\n│  │  ├─ status_registry.json   #mancanti vs completati\n│  │  ├─ embeddings_chunk_i_i+10k.pkl\n│  │  ├─ embeddings_...\n│  │  ├─ db/\n│  │  │  ├─ db_registry.json   #elementi aggiunti al db\n\n'

In [31]:
# import phase
!pip install beir rank_bm25 faiss-cpu ir_measures tqdm lancedb  #hnswlib
#!pip install -U bitsandbytes>=0.46.1
!pip install gptqmodel



In [32]:

from rank_bm25 import BM25Okapi
import numpy as np
import faiss
#import hnswlib
import time
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import torch
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
from datasets import load_dataset
import tqdm

import os

## Model's loading

In [33]:
import os
import time
import torch
import lancedb
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer, CrossEncoder
from huggingface_hub import login
from google.colab import userdata
# 1. Login
TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = TOKEN
login(token=TOKEN)

print("Caricamento modelli in corso...\nExpected Time: 5 min")

print("1. Caricamento Embedder e Reranker...")
bi_enc = SentenceTransformer('BAAI/bge-m3', model_kwargs={"torch_dtype": torch.float16})
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', max_length=1024)

print("2. Caricamento Llama 3.1 8B AWQ...")
model_id = "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Caricamento nativo, sicuro al 100%, niente crash strani
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Caricamento modelli in corso...
Expected Time: 5 min
1. Caricamento Embedder e Reranker...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

2. Caricamento Llama 3.1 8B AWQ...


INFO  Kernel: Auto-selection: adding candidate `AwqExllamaV2Linear`            


INFO  Kernel: selected -> `AwqExllamaV2Linear`.                                


[transformers] Current model requires 3399352832 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.


Loading weights:   0%|          | 0/739 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 28.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 15.81 MiB is free. Including non-PyTorch memory, this process has 14.54 GiB memory in use. Of the allocated memory 14.38 GiB is allocated by PyTorch, and 12.26 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Dataset loading

In [ ]:
# Move vector DB zip file from drive to local disk (move it into the /content/db_local folder)
!cp -r /content/drive/MyDrive/Progetto-NLP/Branch-rag/embedding_collection_ita/db/db_local.zip /content/db_local

# Unzip it into the /content/db_local_extracted folder
!unzip /content/db_local -d /content/db_local_extracted
# 2. Copiamo il file Parquet da Drive all'SSD ultraveloce di Colab
#!cp PARQUET_PATH content/collection.parquet

# Update global variable
VECTOR_DB_PATH = "/content/db_local_extracted/db_local"
#PARQUET_PATH = "/content/collection.parquet"

In [ ]:
# !apt-get install -y pv

# # 1. Create target directories safely
# !mkdir -p /content/db_local /content/db_local_extracted

# # 2. Copy from Drive with a native progress bar
# print("--> Copying zip file from Google Drive...")
# !rsync -ah --progress /content/drive/MyDrive/Progetto-NLP/Branch-rag/embedding_collection_ita/db/db_local.zip /content/db_local/

# # 3. Unzip with a progress bar based on data streaming (Low Overhead)
# print("\n--> Extracting archive contents...")
# !pv /content/db_local/db_local.zip | jar xf /dev/stdin -C /content/db_local_extracted/

In [12]:
import os
import lancedb
#VECTOR_DB_PATH = "/content/db_local_extracted/db_local"

table_name = "wiki_rag_collection"

print("--- DEBUG FILE SYSTEM ---")
if not os.path.exists(VECTOR_DB_PATH):
    print("ERRORE CRITICO: La cartella base non esiste per Colab!")
else:
    contenuto = os.listdir(VECTOR_DB_PATH)
    print(f"Cosa c'è fisicamente dentro '{VECTOR_DB_PATH}':")
    print(contenuto)

    if f"{table_name}.lance" in contenuto:
        print(f"\n✅ PERFETTO! La tabella fisica '{table_name}.lance' C'È.")
    else:
        print(f"\n❌ ERRORE! La tabella fisica '{table_name}.lance' MANCA in questa cartella.")
        print("Probabilmente il path è sbagliato o la cartella è nidificata più a fondo.")

print("\n--- TEST LANCEDB ---")
db = lancedb.connect(VECTOR_DB_PATH)

# TRUCCO: Usa table_names() invece di list_tables()
tabelle_presenti = db.table_names()
print(f"Tabelle viste da LanceDB: {tabelle_presenti}")

if table_name in tabelle_presenti:
    collection = db.open_table(table_name)
    print(f"🎉 SUCCESSO! Tabella '{table_name}' aperta.")
else:
    print(f"❌ FALLIMENTO. LanceDB non vede la tabella.")

--- DEBUG FILE SYSTEM ---
Cosa c'è fisicamente dentro '/content/db_local_extracted/db_local':
['wiki_rag_collection.lance', 'db_registry.json']

✅ PERFETTO! La tabella fisica 'wiki_rag_collection.lance' C'È.

--- TEST LANCEDB ---
Tabelle viste da LanceDB: ['wiki_rag_collection']
🎉 SUCCESSO! Tabella 'wiki_rag_collection' aperta.


/tmp/ipykernel_6637/357918026.py:25: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  tabelle_presenti = db.table_names()


In [13]:
import os
from datasets import load_dataset, load_from_disk

# Assuming DS_PATH (directory for Arrow cache) and PARQUET_PATH are defined earlier

# We save Hugging Face datasets as a directory structure, not a single file
ds_arrow_dir = os.path.join(DS_PATH, "ds_embedding_collection_ita")

ds = None

# Attempt to load from native Hugging Face Disk Cache (Super Fast Arrow Format)
if os.path.exists(ds_arrow_dir):
    print("Attempting to load dataset from native disk cache... \n Expected Time: 5 minutes")
    try:
        ds = load_from_disk(ds_arrow_dir)
        _ = len(ds)  # Quick verification
        print("Dataset loaded successfully from disk cache.")
    except Exception as e:
        print(f"Failed to load dataset from cache ({e}). Attempting to load from raw Parquet instead.")
        ds = None

# Fallback: If cache doesn't exist or is corrupted, load from Parquet
if ds is None:
    if os.path.exists(PARQUET_PATH):
        print("Loading dataset from Parquet...")
        ds = load_dataset("parquet", data_files=PARQUET_PATH, split="train")
        print("Dataset loaded successfully from Parquet.")

        # Save it natively to disk for blazing fast future loading
        print("Caching dataset to disk for future use...")
        ds.save_to_disk(ds_arrow_dir)
        print("Dataset cached successfully.")
    else:
        print(f"Error: Neither cache directory ({ds_arrow_dir}) nor Parquet file ({PARQUET_PATH}) found.")
        raise FileNotFoundError(f"Cannot load dataset. Parquet file not found at {PARQUET_PATH}")

Attempting to load dataset from native disk cache...
Dataset loaded successfully from disk cache.


# Inference

In [34]:
def rag_sota(query, options_text, top_k=2):
    start_time = time.time()
    print("\nInizio retrieval...")

    # A. RETRIEVAL (LanceDB)
    query_vector = bi_enc.encode([query]).tolist()[0]
    end_time_encoding = time.time()
    risultati = collection.search(query_vector).limit(30).to_pandas()
    end_time_index_search = time.time()

    # B. PREPARAZIONE DOCUMENTI
    retrieved_docs = []
    for _, row in risultati.iterrows():
        doc_id = int(row['id'])
        retrieved_docs.append(ds[doc_id]['content'])

    # C. RERANKING
    couples = [[query, doc] for doc in retrieved_docs]
    scores = reranker.predict(couples)

    docs_with_score = list(zip(scores, retrieved_docs))
    docs_with_score.sort(key=lambda x: x[0], reverse=True)

    # D. TOP K DOCS (con TRUNCATION DI SICUREZZA)
    top_docs = [doc for score, doc in docs_with_score[:top_k]]
    docs_context = "\n\n---\n\n".join(top_docs)

    if len(docs_context) > 12000:
        docs_context = docs_context[:12000] + "\n... [TRONCATO PER SICUREZZA]"
        print("TRONCATO")

    print("Retrieval finito.")
    end_time_retrivial = time.time()

    # E. PULIZIA RAM
    del risultati
    del retrieved_docs
    del couples
    #gc.collect()
    torch.cuda.empty_cache()

    # F. PROMPTING
    system_prompt = "You are a quiz solver. Read the context and answer the question"

    user_prompt = f"""<context>
{docs_context}
</context>

Question: {query}
Options: {options_text}

Instructions: The context is in Italian, the options are in English. Analyze the context, reason on its meaning and write ONLY the ID of the correct option. If it's not found, write [NOT FOUND].
Answer:"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    prompt_testo = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # MODIFICA 1: Chiamiamo la variabile 'inputs' e aggiungiamo return_dict=True
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True # <-- FONDAMENTALE
    ).to(model.device)

    # G. INFERENZA NATIVA ULTRA-VELOCE
    outputs = model.generate(
        **inputs,              # <-- MODIFICA 2: Spacchettiamo il dizionario con i due asterischi!
        max_new_tokens=10,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id
    )

    torch.cuda.synchronize()

    # MODIFICA 3: Dobbiamo prendere la lunghezza da inputs['input_ids']
    generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    predicted_answer = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    end_time = time.time()
    tempo_bi_encoding = end_time_encoding - start_time
    tempo_search_index = end_time_index_search - start_time
    tempo_esecuzione = end_time - start_time
    tempo_retrivial = end_time_retrivial - start_time
    print(f"Tempo bi_encodig: {tempo_bi_encoding:.2f} secondi")
    print(f"Tempo ricerca nell'index: {tempo_search_index:.2f} secondi")
    print(f"Tempo retrivial: {tempo_retrivial:.2f} secondi")
    print(f"Tempo di esecuzione totale: {tempo_esecuzione:.2f} secondi")

    return prompt_testo, predicted_answer, top_docs
# ==========================================
# 4. CICLO INFINITO E SALVATAGGIO
# ==========================================

# (Usa la tua vera funzione match_query_options se ne hai una più complessa)
def match_query_options(text):
    if "   [" in text:
        parts = text.split("   [", 1)
        return parts[0].strip(), "[" + parts[1].strip()
    return text, "Nessuna opzione"

os.makedirs("content/", exist_ok=True)
print("\nBot RAG Avviato! (Scrivi 'exit' o 'quit' per fermare il programma)")

while True:
    torch.cuda.empty_cache()
    full_query = input("\n🟢 Inserisci la tua domanda: ").strip()

    if full_query.lower() in ['exit', 'quit']:
        print("Uscita dal programma. A presto!")
        break

    if not full_query:
        continue

    query, options = match_query_options(full_query)
    print(f"Domanda: {query}")
    print(f"Opzioni: {options}")

    # Esecuzione del RAG (Usiamo top_k=2 per bilanciare velocità e precisione)
    prompt1, risposta, documenti_usati = rag_sota(query, options, top_k=2)

    print("\n================== RISPOSTA DEL MODELLO CON CONTESTO ==================")
    print(risposta)

    _, risposta2, _ = rag_sota(query, options, top_k=0)

    print("\n================== RISPOSTA DEL MODELLO CON CONTESTO ==================")
    print(risposta2)

    # Salvataggio nel file (Append mode)
    with open("content/output2.txt", "a", encoding="utf-8") as file:
        file.write("\n\n" + "="*50 + "\n")
        file.write("NUOVA QUERY\n")
        file.write("="*50 + "\n")
        file.write(prompt1)
        file.write("\n\n================== RISPOSTA DEL MODELLO CON CONTESTO ==================\n")
        file.write(risposta)
        file.write("\n\n=======l=========== RISPOSTA DEL MODELLO SENZA CONTESTO ==================\n")
        file.write(risposta2)

    print("[Log salvato in content/output2.txt]")


Bot RAG Avviato! (Scrivi 'exit' o 'quit' per fermare il programma)

🟢 Inserisci la tua domanda: Q: What term describes Whitney Houston's vocal ability to extend notes longer, particularly during slower rearrangements by her band?    [0] Vocal trill   [1] Scat singing   [2] Melisma   [3] Vocal runs
Domanda: Q: What term describes Whitney Houston's vocal ability to extend notes longer, particularly during slower rearrangements by her band?
Opzioni: [0] Vocal trill   [1] Scat singing   [2] Melisma   [3] Vocal runs

Inizio retrieval...


OutOfMemoryError: CUDA out of memory. Tried to allocate 120.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 13.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.39 GiB is allocated by PyTorch, and 13.32 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)